# 单样本相似变换前后可视化

目标：只加载一个 sample，观察相似变换前后的视觉变化。

## 1) 导入依赖与设置随机种子

In [ ]:
import json
import random
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    import cv2
except ImportError as exc:
    raise ImportError('请先安装 opencv-python: pip install opencv-python') from exc

workspace_code = Path('/workspace/code')
if str(workspace_code) not in sys.path:
    sys.path.insert(0, str(workspace_code))

from head3D_fuse.load import load_npz_output

# ------------------------------------------------------------------
# 本 notebook 内部实现：相似变换对齐（不依赖 fuse.py）
# ------------------------------------------------------------------
MIN_POINTS_FOR_ALIGNMENT = 3


def _valid_keypoints_mask(keypoints, zero_eps):
    keypoints = np.asarray(keypoints, dtype=np.float64)
    finite = np.isfinite(keypoints).all(axis=-1)
    nonzero = np.linalg.norm(keypoints, axis=-1) >= zero_eps
    return finite & nonzero


def _resolve_alignment_center(reference, source, pair_valid, center_idx):
    if center_idx is None:
        return None, None
    if center_idx < 0:
        return None, None
    if center_idx >= reference.shape[0] or center_idx >= source.shape[0]:
        return None, None
    if not pair_valid[center_idx]:
        return None, None
    return source[center_idx], reference[center_idx]


def _estimate_similarity_transform(
    source,
    target,
    allow_scale,
    source_center=None,
    target_center=None,
    eps=1e-8,
):
    source = np.asarray(source, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)

    if source_center is None:
        source_center = source.mean(axis=0)
    if target_center is None:
        target_center = target.mean(axis=0)

    source_centered = source - source_center
    target_centered = target - target_center

    h_mat = source_centered.T @ target_centered
    u_mat, s_vals, vt_mat = np.linalg.svd(h_mat)
    rot = vt_mat.T @ u_mat.T
    if np.linalg.det(rot) < 0:
        vt_mat[-1, :] *= -1
        rot = vt_mat.T @ u_mat.T

    scale = 1.0
    if allow_scale:
        denom = float(np.sum(source_centered**2))
        scale = float(np.sum(s_vals) / denom) if denom > eps else 1.0

    translation = target_center - scale * (source_center @ rot)
    return scale, rot, translation


def _finalize_aligned_keypoints(source_points, original_source, valid_mask, scale, rot, translation):
    aligned = (scale * source_points @ rot) + translation
    aligned[~valid_mask] = original_source[~valid_mask]
    return aligned


def _select_trimmed_inliers(residuals, valid_mask, trim_ratio):
    if trim_ratio <= 0:
        return valid_mask
    if trim_ratio >= 1.0:
        raise ValueError(f'trim_ratio must be < 1.0, got {trim_ratio}')

    valid_idx = np.flatnonzero(valid_mask)
    n_valid = valid_idx.size
    if n_valid == 0:
        return valid_mask

    raw_keep = int(np.ceil((1.0 - trim_ratio) * n_valid))
    n_keep = min(n_valid, max(MIN_POINTS_FOR_ALIGNMENT, raw_keep))
    order = np.argsort(residuals[valid_idx])
    keep_idx = valid_idx[order[:n_keep]]

    trimmed = np.zeros_like(valid_mask, dtype=bool)
    trimmed[keep_idx] = True
    return trimmed


def _align_keypoints_to_reference(reference, source, zero_eps=1e-6, allow_scale=True, center_idx=None):
    ref = np.asarray(reference, dtype=np.float64)
    src = np.asarray(source, dtype=np.float64)

    ref_valid = _valid_keypoints_mask(ref, zero_eps)
    src_valid = _valid_keypoints_mask(src, zero_eps)
    pair_valid = ref_valid & src_valid

    if pair_valid.sum() < MIN_POINTS_FOR_ALIGNMENT:
        return source

    source_center, target_center = _resolve_alignment_center(ref, src, pair_valid, center_idx)
    scale, rot, translation = _estimate_similarity_transform(
        src[pair_valid],
        ref[pair_valid],
        allow_scale,
        source_center=source_center,
        target_center=target_center,
    )
    return _finalize_aligned_keypoints(src, source, src_valid, scale, rot, translation)


def _align_keypoints_trimmed(
    reference,
    source,
    zero_eps=1e-6,
    allow_scale=True,
    trim_ratio=0.2,
    max_iters=3,
    center_idx=None,
):
    ref = np.asarray(reference, dtype=np.float64)
    src = np.asarray(source, dtype=np.float64)

    ref_valid = _valid_keypoints_mask(ref, zero_eps)
    src_valid = _valid_keypoints_mask(src, zero_eps)
    pair_valid = ref_valid & src_valid

    if pair_valid.sum() < MIN_POINTS_FOR_ALIGNMENT:
        return source

    source_center, target_center = _resolve_alignment_center(ref, src, pair_valid, center_idx)

    inlier_mask = pair_valid.copy()
    scale = 1.0
    rot = np.eye(3)
    translation = np.zeros(3)

    for _ in range(max_iters):
        if inlier_mask.sum() < MIN_POINTS_FOR_ALIGNMENT:
            break

        scale, rot, translation = _estimate_similarity_transform(
            src[inlier_mask],
            ref[inlier_mask],
            allow_scale,
            source_center=source_center,
            target_center=target_center,
        )
        aligned = (scale * src @ rot) + translation
        residuals = np.linalg.norm(aligned - ref, axis=-1)
        new_inlier_mask = _select_trimmed_inliers(residuals, pair_valid, trim_ratio)

        if np.array_equal(new_inlier_mask, inlier_mask):
            break
        inlier_mask = new_inlier_mask

    if inlier_mask.sum() < MIN_POINTS_FOR_ALIGNMENT:
        return source

    return _finalize_aligned_keypoints(src, source, src_valid, scale, rot, translation)


ALIGN_SUPPORTS_CENTER = True
print('ALIGN_SUPPORTS_CENTER:', ALIGN_SUPPORTS_CENTER)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['image.cmap'] = 'gray'
print('Seed:', SEED)

## 2) 配置数据路径并仅加载一个 Sample

In [ ]:
# 参数区
person_id = '01'
env_name = '昼多い'
views = ['front', 'left', 'right']

data_root = Path('/workspace/data/sam3d_body_results_right_full')
sample_root = data_root / person_id / env_name

# 与 Procrustes 3D 对齐相关参数
zero_eps = 1e-6
allow_scale = False
trim_ratio = 0.2
max_iters = 3

# 图像与2D关键点相似变换参数
scale = 1.1
angle_deg = 15.0
tx, ty = 20.0, -10.0


def extract_frame_idx(npz_path: Path):
    # 注意: 这里必须是 r'^(\d+)' 的语义（匹配文件名前导数字）。
    # 在 notebook JSON 里写成字符串时只需保留一个反斜杠给正则: r'^(\d+)'
    m = re.match(r'^(\d+)', npz_path.stem)
    return int(m.group(1)) if m else None


def normalize_keypoints(arr):
    if arr is None:
        return None
    arr = np.asarray(arr)
    if arr.ndim == 3 and arr.shape[0] >= 1:
        arr = arr[0]
    return arr


def build_frame_map(view_dir: Path):
    frame_map = {}
    for p in sorted(view_dir.glob('*.npz')):
        idx = extract_frame_idx(p)
        if idx is not None:
            frame_map[idx] = p
    return frame_map


def collect_common_frames(root_dir: Path, view_names):
    maps = {v: build_frame_map(root_dir / v) for v in view_names}
    if not maps:
        return maps, []
    common = sorted(set.intersection(*[set(m.keys()) for m in maps.values()]))
    return maps, common


# 先尝试用户指定环境
frame_maps, common_frames = collect_common_frames(sample_root, views)

# 若无共同帧，自动回退到该 person 下第一个可用环境
if not common_frames:
    person_root = data_root / person_id
    candidate_envs = [p for p in sorted(person_root.iterdir()) if p.is_dir()]
    selected_env = None
    for env_dir in candidate_envs:
        fm, cf = collect_common_frames(env_dir, views)
        if cf:
            selected_env = env_dir.name
            env_name = env_dir.name
            sample_root = env_dir
            frame_maps, common_frames = fm, cf
            break

    if selected_env is None:
        debug_counts = {
            v: len(build_frame_map(sample_root / v)) for v in views
        }
        raise RuntimeError(
            f'在 {person_root} 下未找到任何包含三视角共同帧的环境。'
            f' 当前环境 {sample_root} 的每视角帧数: {debug_counts}'
        )
    else:
        print(f'原环境无共同帧，已自动切换到: {env_name}')

frame_idx = common_frames[0]  # 只加载一个 sample
npz_paths = {v: frame_maps[v][frame_idx] for v in views}
outputs = {v: load_npz_output(p) for v, p in npz_paths.items()}

print('Selected sample metadata:')
print('  person_id:', person_id)
print('  env_name :', env_name)
print('  frame_idx:', frame_idx)
for v, p in npz_paths.items():
    print(f'  {v:>5}: {p.name}')

In [ ]:
# 定义需要保留的关键点索引：头部 + 肩部/颈部
KEEP_KEYPOINT_INDICES = (
    # 头部: 鼻子、眼睛、耳朵
    list(range(0, 5))  # 0-4: nose, left-eye, right-eye, left-ear, right-ear
    # 肩部和颈部
    + [5, 6]  # left-shoulder, right-shoulder
    # 肩峰和颈部
    + [67, 68, 69]  # left-acromion, right-acromion, neck
)


## 3) 读取图像与关键点并做原始可视化

In [ ]:
def get_frame_image(output_dict):
    frame = output_dict.get('frame')
    if frame is None:
        raise KeyError('npz output 缺少 frame 字段')
    frame = np.asarray(frame)
    if frame.ndim == 2:
        frame = np.stack([frame] * 3, axis=-1)
    return frame


def get_2d_keypoints(output_dict):
    kpts2d = normalize_keypoints(output_dict.get('pred_keypoints_2d'))
    if kpts2d is None:
        raise KeyError('npz output 缺少 pred_keypoints_2d 字段')
    kpts2d = np.asarray(kpts2d)
    if kpts2d.shape[-1] < 2:
        raise ValueError('pred_keypoints_2d 最后一个维度应 >= 2')
    return kpts2d[:, :2]


def get_3d_keypoints(output_dict):
    kpts3d = normalize_keypoints(output_dict.get('pred_keypoints_3d'))
    if kpts3d is None:
        raise KeyError('npz output 缺少 pred_keypoints_3d 字段')
    kpts3d = np.asarray(kpts3d)
    if kpts3d.shape[-1] < 3:
        raise ValueError('pred_keypoints_3d 最后一个维度应 >= 3')
    return kpts3d[:, :3]


def filter_keypoints_by_indices(kpts, keep_indices):
    kpts = np.asarray(kpts)
    valid_idx = [i for i in keep_indices if 0 <= i < kpts.shape[0]]
    if not valid_idx:
        raise ValueError('过滤索引为空，请检查 KEEP_KEYPOINT_INDICES')
    return kpts[valid_idx], valid_idx

sample = {}
active_keep_indices = None
for v in views:
    frame = get_frame_image(outputs[v])
    kpts2d = get_2d_keypoints(outputs[v])
    kpts3d = get_3d_keypoints(outputs[v])

    kpts2d_f, used_idx = filter_keypoints_by_indices(kpts2d, KEEP_KEYPOINT_INDICES)
    kpts3d_f, _ = filter_keypoints_by_indices(kpts3d, used_idx)

    if active_keep_indices is None:
        active_keep_indices = used_idx

    sample[v] = {
        'frame': frame,
        'kpts2d': kpts2d_f,
        'kpts3d': kpts3d_f,
    }

print(f'已过滤关键点数量: {len(active_keep_indices)} / 原始数量: {kpts2d.shape[0]}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, v in zip(axes, views):
    img = sample[v]['frame']
    kpts = sample[v]['kpts2d']
    valid = np.isfinite(kpts).all(axis=1)

    ax.imshow(img)
    ax.scatter(kpts[valid, 0], kpts[valid, 1], s=14, c='lime', label='2D kpts (filtered)')
    ax.set_title(f'Raw view: {v}')
    ax.axis('off')
    ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

## 3.1) Three-View Image Overview

Show the front, left, and right views side by side with filtered 2D keypoints.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

for ax, v in zip(axes, views):
    img = sample[v]['frame']
    kpts = sample[v]['kpts2d']
    valid = np.isfinite(kpts).all(axis=1)

    ax.imshow(img)
    ax.scatter(kpts[valid, 0], kpts[valid, 1], s=14, c='lime', label='2D kpts (filtered)')
    ax.set_title(f'{v.capitalize()} view')
    ax.axis('off')
    ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

## 4) 定义相似变换参数与变换函数

In [ ]:
def build_similarity_affine(shape_hw, scale=1.0, angle_deg=0.0, tx=0.0, ty=0.0):
    h, w = shape_hw[:2]
    cx, cy = (w - 1) * 0.5, (h - 1) * 0.5
    mat = cv2.getRotationMatrix2D((cx, cy), angle_deg, scale)
    mat[:, 2] += np.array([tx, ty], dtype=np.float64)
    return mat


def warp_image(image, affine_mat, interpolation=cv2.INTER_LINEAR):
    h, w = image.shape[:2]
    return cv2.warpAffine(image, affine_mat, (w, h), flags=interpolation, borderMode=cv2.BORDER_REFLECT)


def transform_keypoints_2d(kpts2d, affine_mat):
    kpts = np.asarray(kpts2d, dtype=np.float64)
    homo = np.concatenate([kpts, np.ones((kpts.shape[0], 1), dtype=np.float64)], axis=1)
    out = (affine_mat @ homo.T).T
    invalid = ~np.isfinite(kpts).all(axis=1)
    out[invalid] = np.nan
    return out

## 5) 对单个 Sample 应用相似变换

In [ ]:
target_view = 'front'
raw_img = sample[target_view]['frame']
raw_kpts2d = sample[target_view]['kpts2d']

affine_mat = build_similarity_affine(raw_img.shape, scale=scale, angle_deg=angle_deg, tx=tx, ty=ty)
transformed_img = warp_image(raw_img, affine_mat)
transformed_kpts2d = transform_keypoints_2d(raw_kpts2d, affine_mat)

print('Affine matrix (2x3):')
print(affine_mat)

## 附加：3D Procrustes 相似变换前后效果（与 front 对齐）

In [ ]:
def mean_dist(a, b):
    valid = np.isfinite(a).all(axis=1) & np.isfinite(b).all(axis=1)
    if not np.any(valid):
        return np.nan
    return float(np.mean(np.linalg.norm(a[valid] - b[valid], axis=1)))


front3d = np.asarray(sample['front']['kpts3d'], dtype=np.float64)
left3d = np.asarray(sample['left']['kpts3d'], dtype=np.float64)
right3d = np.asarray(sample['right']['kpts3d'], dtype=np.float64)


def align_to_ref(reference, source, trimmed=False):
    kwargs = dict(zero_eps=zero_eps, allow_scale=allow_scale)
    if trimmed:
        kwargs.update(trim_ratio=trim_ratio, max_iters=max_iters)

    if trimmed:
        return _align_keypoints_trimmed(reference, source, **kwargs)
    return _align_keypoints_to_reference(reference, source, **kwargs)


left_pro = align_to_ref(front3d, left3d, trimmed=False)
right_pro = align_to_ref(front3d, right3d, trimmed=False)
left_trim = align_to_ref(front3d, left3d, trimmed=True)
right_trim = align_to_ref(front3d, right3d, trimmed=True)

print('Mean distance to front (left): before / procrustes / trimmed =',
      f"{mean_dist(front3d, left3d):.6f}",
      f"{mean_dist(front3d, left_pro):.6f}",
      f"{mean_dist(front3d, left_trim):.6f}")
print('Mean distance to front (right): before / procrustes / trimmed =',
      f"{mean_dist(front3d, right3d):.6f}",
      f"{mean_dist(front3d, right_pro):.6f}",
      f"{mean_dist(front3d, right_trim):.6f}")


def set_equal_3d_axes(ax, xyz_list):
    all_pts = np.concatenate([p[np.isfinite(p).all(axis=1)] for p in xyz_list], axis=0)
    mins = all_pts.min(axis=0)
    maxs = all_pts.max(axis=0)
    center = (mins + maxs) / 2.0
    radius = (maxs - mins).max() / 2.0
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)


fig = plt.figure(figsize=(18, 5))
ax1 = fig.add_subplot(131, projection='3d')
ax2 = fig.add_subplot(132, projection='3d')
ax3 = fig.add_subplot(133, projection='3d')

for ax in [ax1, ax2, ax3]:
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

ax1.scatter(front3d[:, 0], front3d[:, 1], front3d[:, 2], s=16, c='black', label='front')
ax1.scatter(left3d[:, 0], left3d[:, 1], left3d[:, 2], s=16, c='tab:blue', label='left raw')
ax1.scatter(right3d[:, 0], right3d[:, 1], right3d[:, 2], s=16, c='tab:red', label='right raw')
ax1.set_title('Before 3D Alignment')

ax2.scatter(front3d[:, 0], front3d[:, 1], front3d[:, 2], s=16, c='black', label='front')
ax2.scatter(left_pro[:, 0], left_pro[:, 1], left_pro[:, 2], s=16, c='tab:blue', label='left procrustes')
ax2.scatter(right_pro[:, 0], right_pro[:, 1], right_pro[:, 2], s=16, c='tab:red', label='right procrustes')
ax2.set_title('After Procrustes')

ax3.scatter(front3d[:, 0], front3d[:, 1], front3d[:, 2], s=16, c='black', label='front')
ax3.scatter(left_trim[:, 0], left_trim[:, 1], left_trim[:, 2], s=16, c='tab:blue', label='left trimmed')
ax3.scatter(right_trim[:, 0], right_trim[:, 1], right_trim[:, 2], s=16, c='tab:red', label='right trimmed')
ax3.set_title('After Trimmed Procrustes')

set_equal_3d_axes(ax1, [front3d, left3d, right3d])
set_equal_3d_axes(ax2, [front3d, left_pro, right_pro])
set_equal_3d_axes(ax3, [front3d, left_trim, right_trim])

for ax in [ax1, ax2, ax3]:
    ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

## 9) 融合之后的效果对比图（Raw vs Procrustes vs Trimmed）

In [ ]:
def fuse_median_points(*arrays):
    stacked = np.stack(arrays, axis=0).astype(np.float64)
    valid = np.isfinite(stacked).all(axis=-1)
    fused = np.full(stacked.shape[1:], np.nan, dtype=np.float64)
    if valid.any():
        slice_data = stacked.copy()
        slice_data[~valid] = np.nan
        fused = np.nanmedian(slice_data, axis=0)
    return fused


if 'left_pro' not in globals() or 'right_pro' not in globals() or 'left_trim' not in globals() or 'right_trim' not in globals():
    front3d = np.asarray(sample['front']['kpts3d'], dtype=np.float64)
    left3d = np.asarray(sample['left']['kpts3d'], dtype=np.float64)
    right3d = np.asarray(sample['right']['kpts3d'], dtype=np.float64)

    left_pro = _align_keypoints_to_reference(front3d, left3d, zero_eps=zero_eps, allow_scale=allow_scale)
    right_pro = _align_keypoints_to_reference(front3d, right3d, zero_eps=zero_eps, allow_scale=allow_scale)
    left_trim = _align_keypoints_trimmed(front3d, left3d, zero_eps=zero_eps, allow_scale=allow_scale, trim_ratio=trim_ratio, max_iters=max_iters)
    right_trim = _align_keypoints_trimmed(front3d, right3d, zero_eps=zero_eps, allow_scale=allow_scale, trim_ratio=trim_ratio, max_iters=max_iters)

fused_raw = fuse_median_points(front3d, left3d, right3d)
fused_pro = fuse_median_points(front3d, left_pro, right_pro)
fused_trim = fuse_median_points(front3d, left_trim, right_trim)

print('Mean diff fused_raw -> fused_pro:', f"{mean_dist(fused_raw, fused_pro):.6f}")
print('Mean diff fused_raw -> fused_trim:', f"{mean_dist(fused_raw, fused_trim):.6f}")
print('Mean diff fused_pro -> fused_trim:', f"{mean_dist(fused_pro, fused_trim):.6f}")

fig = plt.figure(figsize=(18, 5))
ax1 = fig.add_subplot(131, projection='3d')
ax2 = fig.add_subplot(132, projection='3d')
ax3 = fig.add_subplot(133, projection='3d')

# Raw fusion (gray)
ax1.scatter(fused_raw[:, 0], fused_raw[:, 1], fused_raw[:, 2], s=24, c='gray', alpha=0.9, label='Fused Result', edgecolors='darkgray', linewidth=0.5)
ax1.scatter(front3d[:, 0], front3d[:, 1], front3d[:, 2], s=8, c='black', alpha=0.3, label='Front Ref')
ax1.set_title('Raw Views Fusion\n(No Alignment)', fontsize=12, fontweight='bold')

# Procrustes fusion (blue)
ax2.scatter(fused_pro[:, 0], fused_pro[:, 1], fused_pro[:, 2], s=24, c='dodgerblue', alpha=0.9, label='Fused Result', edgecolors='navy', linewidth=0.5)
ax2.scatter(front3d[:, 0], front3d[:, 1], front3d[:, 2], s=8, c='black', alpha=0.3, label='Front Ref')
ax2.set_title('Procrustes Alignment\n(Rigid Transform)', fontsize=12, fontweight='bold')

# Trimmed Procrustes fusion (red)
ax3.scatter(fused_trim[:, 0], fused_trim[:, 1], fused_trim[:, 2], s=24, c='orangered', alpha=0.9, label='Fused Result', edgecolors='darkred', linewidth=0.5)
ax3.scatter(front3d[:, 0], front3d[:, 1], front3d[:, 2], s=8, c='black', alpha=0.3, label='Front Ref')
ax3.set_title('Trimmed Procrustes\n(Robust Alignment)', fontsize=12, fontweight='bold')

set_equal_3d_axes(ax1, [fused_raw, front3d])
set_equal_3d_axes(ax2, [fused_pro, front3d])
set_equal_3d_axes(ax3, [fused_trim, front3d])

for ax in [ax1, ax2, ax3]:
    ax.set_xlabel('X', fontsize=10)
    ax.set_ylabel('Y', fontsize=10)
    ax.set_zlabel('Z', fontsize=10)
    ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

## 10) 三个手法 × 三个视角 对比

- 三个手法: Raw / Procrustes / Trimmed Procrustes
- 三个视角: front / left / right
- 指标: 每个手法的融合结果与各视角关键点的平均欧氏距离（越小越好）

In [ ]:
# Auto-fill upstream variables if they don't exist
if 'front3d' not in globals():
    front3d = np.asarray(sample['front']['kpts3d'], dtype=np.float64)
    left3d = np.asarray(sample['left']['kpts3d'], dtype=np.float64)
    right3d = np.asarray(sample['right']['kpts3d'], dtype=np.float64)

if 'left_pro' not in globals() or 'right_pro' not in globals() or 'left_trim' not in globals() or 'right_trim' not in globals():
    left_pro = _align_keypoints_to_reference(front3d, left3d, zero_eps=zero_eps, allow_scale=allow_scale)
    right_pro = _align_keypoints_to_reference(front3d, right3d, zero_eps=zero_eps, allow_scale=allow_scale)
    left_trim = _align_keypoints_trimmed(front3d, left3d, zero_eps=zero_eps, allow_scale=allow_scale, trim_ratio=trim_ratio, max_iters=max_iters)
    right_trim = _align_keypoints_trimmed(front3d, right3d, zero_eps=zero_eps, allow_scale=allow_scale, trim_ratio=trim_ratio, max_iters=max_iters)

if 'fused_raw' not in globals() or 'fused_pro' not in globals() or 'fused_trim' not in globals():
    fused_raw = fuse_median_points(front3d, left3d, right3d)
    fused_pro = fuse_median_points(front3d, left_pro, right_pro)
    fused_trim = fuse_median_points(front3d, left_trim, right_trim)


def mean_dist_nan_robust(a, b):
    valid = np.isfinite(a).all(axis=1) & np.isfinite(b).all(axis=1)
    if not np.any(valid):
        return np.nan
    return float(np.mean(np.linalg.norm(a[valid] - b[valid], axis=1)))


methods = {
    'Raw': {
        'fused': fused_raw,
        'views': {'front': front3d, 'left': left3d, 'right': right3d},
    },
    'Procrustes': {
        'fused': fused_pro,
        'views': {'front': front3d, 'left': left_pro, 'right': right_pro},
    },
    'Trimmed': {
        'fused': fused_trim,
        'views': {'front': front3d, 'left': left_trim, 'right': right_trim},
    },
}

# 1) Numerical comparison table
print('Mean distance: fused(method) -> view (lower is better)')
header = f"{'Method':<12} {'front':>12} {'left':>12} {'right':>12}"
print(header)
print('-' * len(header))
for m_name, m_data in methods.items():
    d_front = mean_dist_nan_robust(m_data['fused'], m_data['views']['front'])
    d_left = mean_dist_nan_robust(m_data['fused'], m_data['views']['left'])
    d_right = mean_dist_nan_robust(m_data['fused'], m_data['views']['right'])
    print(f"{m_name:<12} {d_front:12.6f} {d_left:12.6f} {d_right:12.6f}")

# 2) 3x3 visualization grid: rows = methods, columns = views
fig = plt.figure(figsize=(16, 14))
row_names = list(methods.keys())
col_names = ['front', 'left', 'right']

# Define colors for each method
method_colors = {
    'Raw': ('darkgray', 'lightgray', 'gray'),         # (fused, view, label)
    'Procrustes': ('dodgerblue', 'lightblue', 'navy'),
    'Trimmed': ('orangered', 'lightsalmon', 'darkred'),
}

for r, m_name in enumerate(row_names):
    fused = methods[m_name]['fused']
    fused_color, view_color, edge_color = method_colors[m_name]
    
    for c, v_name in enumerate(col_names):
        ax = fig.add_subplot(3, 3, r * 3 + c + 1, projection='3d')
        view_pts = methods[m_name]['views'][v_name]

        # Original view point cloud (light color)
        ax.scatter(view_pts[:, 0], view_pts[:, 1], view_pts[:, 2], 
                  s=10, c=view_color, alpha=0.6, label=f'{v_name.upper()} view', edgecolors='none')
        
        # Fused result (dark color with border)
        ax.scatter(fused[:, 0], fused[:, 1], fused[:, 2], 
                  s=20, c=fused_color, alpha=0.95, label='Fused', edgecolors=edge_color, linewidth=0.5)

        dist = mean_dist_nan_robust(fused, view_pts)
        ax.set_title(f'{m_name} × {v_name.capitalize()}\nDistance={dist:.4f}', fontsize=11, fontweight='bold')
        ax.set_xlabel('X', fontsize=9)
        ax.set_ylabel('Y', fontsize=9)
        ax.set_zlabel('Z', fontsize=9)
        set_equal_3d_axes(ax, [fused, view_pts])
        ax.legend(loc='upper left', fontsize=9)
        
        # Add grid
        ax.grid(True, alpha=0.3)

# Overall title
fig.suptitle('Fusion Comparison: 3 Methods × 3 Views (Colors: Gray/Blue/Red = Raw/Procrustes/Trimmed)', 
             fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 3D keypoints of three views in one shared coordinate system
# with semantic point names and three camera angles
if 'front3d' not in globals() or 'left3d' not in globals() or 'right3d' not in globals():
    front3d = np.asarray(sample['front']['kpts3d'], dtype=np.float64)
    left3d = np.asarray(sample['left']['kpts3d'], dtype=np.float64)
    right3d = np.asarray(sample['right']['kpts3d'], dtype=np.float64)

# Semantic names for original keypoint indices used in this notebook
KEYPOINT_NAME_MAP = {
    0: 'nose',
    1: 'left_eye',
    2: 'right_eye',
    3: 'left_ear',
    4: 'right_ear',
    5: 'left_shoulder',
    6: 'right_shoulder',
    67: 'left_acromion',
    68: 'right_acromion',
    69: 'neck',
}

n_pts = front3d.shape[0]
if 'active_keep_indices' in globals() and active_keep_indices is not None and len(active_keep_indices) == n_pts:
    point_names = [KEYPOINT_NAME_MAP.get(idx, f'idx{idx}') for idx in active_keep_indices]
else:
    point_names = [f'k{i}' for i in range(n_pts)]

# Swap Y and Z: (x, y, z) -> (x, z, y)
front_plot = np.stack([front3d[:, 0], front3d[:, 2], front3d[:, 1]], axis=1)
left_plot = np.stack([left3d[:, 0], left3d[:, 2], left3d[:, 1]], axis=1)
right_plot = np.stack([right3d[:, 0], right3d[:, 2], right3d[:, 1]], axis=1)

view_settings = [
    ('Perspective', 25, -55),
    ('Side View', 10, 0),
    ('Top View', 90, -90),
]

fig = plt.figure(figsize=(18, 6))
for plot_idx, (view_name, elev, azim) in enumerate(view_settings, start=1):
    ax = fig.add_subplot(1, 3, plot_idx, projection='3d')

    ax.scatter(front_plot[:, 0], front_plot[:, 1], front_plot[:, 2], s=30, c='black', alpha=0.9, label='Front')
    ax.scatter(left_plot[:, 0], left_plot[:, 1], left_plot[:, 2], s=30, c='tab:blue', alpha=0.9, label='Left')
    ax.scatter(right_plot[:, 0], right_plot[:, 1], right_plot[:, 2], s=30, c='tab:red', alpha=0.9, label='Right')

    # Keep labels on first panel to reduce clutter
    if plot_idx == 1:
        for i, name in enumerate(point_names):
            if np.isfinite(front_plot[i]).all():
                ax.text(
                    front_plot[i, 0] + 0.001,
                    front_plot[i, 1] + 0.001,
                    front_plot[i, 2] + 0.001,
                    f'{name}_F',
                    fontsize=7,
                    color='black',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.55, pad=0.15),
                )
            if np.isfinite(left_plot[i]).all():
                ax.text(
                    left_plot[i, 0] + 0.001,
                    left_plot[i, 1] + 0.001,
                    left_plot[i, 2] + 0.001,
                    f'{name}_L',
                    fontsize=7,
                    color='tab:blue',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.55, pad=0.15),
                )
            if np.isfinite(right_plot[i]).all():
                ax.text(
                    right_plot[i, 0] + 0.001,
                    right_plot[i, 1] + 0.001,
                    right_plot[i, 2] + 0.001,
                    f'{name}_R',
                    fontsize=7,
                    color='tab:red',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.55, pad=0.15),
                )

    set_equal_3d_axes(ax, [front_plot, left_plot, right_plot])
    ax.view_init(elev=elev, azim=azim)
    ax.set_title(view_name, fontsize=12, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Z (original)')
    ax.set_zlabel('Y (original)')
    ax.grid(True, alpha=0.3)
    if plot_idx == 1:
        ax.legend(loc='upper left')

fig.suptitle('Three-View 3D Keypoints (Y/Z Swapped) from Three Camera Angles', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 12) Pre-Fusion KPTs After Rigid Alignment

Show 3D keypoints after rigid alignment (rotation + translation, no scaling) before fusion.

In [ ]:
# Pre-fusion visualization after rigid alignment (no scaling)
if 'front3d' not in globals() or 'left3d' not in globals() or 'right3d' not in globals():
    front3d = np.asarray(sample['front']['kpts3d'], dtype=np.float64)
    left3d = np.asarray(sample['left']['kpts3d'], dtype=np.float64)
    right3d = np.asarray(sample['right']['kpts3d'], dtype=np.float64)

# Rigid alignment only: allow_scale=False
left_rigid = _align_keypoints_to_reference(
    front3d,
    left3d,
    zero_eps=zero_eps,
    allow_scale=False,
)
right_rigid = _align_keypoints_to_reference(
    front3d,
    right3d,
    zero_eps=zero_eps,
    allow_scale=False,
)

print('Mean distance to front (before -> after rigid):')
print('  left :', f"{mean_dist(front3d, left3d):.6f}", '->', f"{mean_dist(front3d, left_rigid):.6f}")
print('  right:', f"{mean_dist(front3d, right3d):.6f}", '->', f"{mean_dist(front3d, right_rigid):.6f}")

# Semantic names for all points
KEYPOINT_NAME_MAP = {
    0: 'nose',
    1: 'left_eye',
    2: 'right_eye',
    3: 'left_ear',
    4: 'right_ear',
    5: 'left_shoulder',
    6: 'right_shoulder',
    67: 'left_acromion',
    68: 'right_acromion',
    69: 'neck',
}

if 'active_keep_indices' in globals() and active_keep_indices is not None and len(active_keep_indices) == front3d.shape[0]:
    labels = [KEYPOINT_NAME_MAP.get(idx, f'idx{idx}') for idx in active_keep_indices]
else:
    labels = [f'k{i}' for i in range(front3d.shape[0])]

# Three camera views for better 3D inspection
view_settings = [
    ('Perspective', 25, -55),
    ('Side View', 10, 0),
    ('Top View', 90, -90),
]

fig = plt.figure(figsize=(18, 6))
for plot_idx, (view_name, elev, azim) in enumerate(view_settings, start=1):
    ax = fig.add_subplot(1, 3, plot_idx, projection='3d')

    # Pre-fusion points only (no fused points)
    ax.scatter(front3d[:, 0], front3d[:, 1], front3d[:, 2], s=28, c='black', alpha=0.9, label='Front (reference)')
    ax.scatter(left_rigid[:, 0], left_rigid[:, 1], left_rigid[:, 2], s=28, c='tab:blue', alpha=0.9, label='Left after rigid')
    ax.scatter(right_rigid[:, 0], right_rigid[:, 1], right_rigid[:, 2], s=28, c='tab:red', alpha=0.9, label='Right after rigid')

    # Keep labels on the first panel only to avoid severe overlap
    if plot_idx == 1:
        for i, name in enumerate(labels):
            if np.isfinite(front3d[i]).all():
                ax.text(
                    front3d[i, 0] + 0.001,
                    front3d[i, 1] + 0.001,
                    front3d[i, 2] + 0.001,
                    f'{name}_F',
                    fontsize=7,
                    color='black',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.55, pad=0.15),
                )
            if np.isfinite(left_rigid[i]).all():
                ax.text(
                    left_rigid[i, 0] + 0.001,
                    left_rigid[i, 1] + 0.001,
                    left_rigid[i, 2] + 0.001,
                    f'{name}_L',
                    fontsize=7,
                    color='tab:blue',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.55, pad=0.15),
                )
            if np.isfinite(right_rigid[i]).all():
                ax.text(
                    right_rigid[i, 0] + 0.001,
                    right_rigid[i, 1] + 0.001,
                    right_rigid[i, 2] + 0.001,
                    f'{name}_R',
                    fontsize=7,
                    color='tab:red',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.55, pad=0.15),
                )

    set_equal_3d_axes(ax, [front3d, left_rigid, right_rigid])
    ax.view_init(elev=elev, azim=azim)
    ax.set_title(f'{view_name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.grid(True, alpha=0.3)
    if plot_idx == 1:
        ax.legend(loc='upper left')

fig.suptitle('Pre-Fusion 3D KPTs After Rigid Alignment (Three View Angles)', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()